In [51]:
%pip install langchain faiss-cpu sentence-transformers transformers accelerate
%pip install -U langchain langchain-community
%pip install pypdf
%pip install hf_xet
%pip install -U langchain-huggingface

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

import re
from sentence_transformers import CrossEncoder

c:\Users\avvaic\CODE\DomainSpecificLLMs\DomainSpecificLLMs-1\DSL\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Load PDf

In [ ]:
# Load PDF
loader = PyPDFLoader("./Docs/France.pdf")  # Replace with your PDF path
docs = loader.load()
docs = docs[1:]

### 1.5 Pre-Process Text

In [4]:
def clean_text(text):
    # Replace newlines followed by lowercase letter with space (joining broken sentences)
    text = re.sub(r'\n(?=[a-z])', ' ', text)
    # Replace multiple spaces/newlines with single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


for doc in docs:
    doc.page_content = clean_text(doc.page_content)


### 2. Chunk Text with Overlap

In [5]:
# Chunk text with overlap (default chunk size ~1000 chars, overlap ~200)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(chunks)}")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ---")
    print(chunk.page_content[:500])  # print first 500 chars to keep it manageable
    print()


Total chunks created: 1133
--- Chunk 0 ---
1 CHAPTER ONE OWL POST Harry Potter was a highly unusual boy in many ways. For one thing, he hated the summer holidays more than any other time of year. For another, he really wanted to do his homework but was forced to do it in secret, in the dead of night. And he also happened to be a wizard. It was nearly midnight, and he was lying on his stomach in bed, the blankets drawn right over his head like a tent, a flashlight in one hand and a large leather-bound book (A History of Magic by Bathilda 

--- Chunk 1 ---
frowning as he looked for something that would help him write his essay, "Witch Burning in the Fourteenth Century Was Completely Pointless discuss." The quill paused at the top of a likely-looking paragraph. Harry Pushed his round glasses up the bridge of his nose, moved his flashlight closer to the book, and read: Non-magic people (more commonly known as Muggles) were particularly afraid of magic in medieval times, but not very good a

### 3. Create Embeddings Locally with Sentence Transformers

In [6]:
# Create embeddings locally with SentenceTransformers
embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")

C:\Users\avvaic\AppData\Local\Temp\ipykernel_18748\2588165994.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")


### 4. Build FAISS index

In [7]:
# Build FAISS vector store from chunks
chunks = [doc for doc in chunks if not doc.page_content.strip().startswith(tuple("0123456789")) and len(doc.page_content) > 100]
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


KeyboardInterrupt: 

### 5. Set up Local LLM 

In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

tokenizer.model_max_length = 512
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_new_tokens=200)
llm = HuggingFacePipeline(pipeline=pipe)

Device set to use cpu


### 6. Create RetrievalQA Chain

In [ ]:
prompt = PromptTemplate.from_template(
    "Answer the following question based only on the context provided.\n\nContext:\n{context}\n\nQuestion: {question}"
)


def get_context_text(docs, max_tokens=800):
    texts = [doc.page_content for doc in docs]
    tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    joined = ""
    total_tokens = 0
    for text in texts:
        tokens = tokenizer.encode(text, truncation=False, add_special_tokens=False)
        if total_tokens + len(tokens) > max_tokens:
            break
        joined += text + "\n\n"
        total_tokens += len(tokens)
    return joined.strip()


reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(question, docs):
    pairs = [(question, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)
    scored_docs = list(zip(docs, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, score in scored_docs[:5]]



def ask(question, show_chunks=True):
    docs_with_scores = vectorstore.search(question, search_type="similarity", k=5)

    docs_reranked = rerank(question, docs_with_scores)
    context = get_context_text(docs_reranked)


    prompt_str = prompt.format(context=context, question=question)
    if show_chunks:
        print("\n--- Retrieved Chunks ---")
        for i, doc in enumerate(docs_reranked):
            print(f"\n[Chunk {i+1}]\n{doc.page_content[:500]}...")  # show first 500 chars
            print(f"\n[Chunk {i+1}] Score: {doc.metadata.get('score', 'N/A')}\n{doc.page_content[:400]}")
        print("\n------------------------")

    return llm.invoke(prompt_str)


### 7. Ask Questions

In [ ]:
print(ask("What did France do during WW2"))

Token indices sequence length is longer than the specified maximum sequence length for this model (672 > 512). Running this sequence through the model will result in indexing errors



--- Retrieved Chunks ---

[Chunk 1]
In 1940, France was invaded and quickly defeated by Nazi Germany. France was divided into a German occupation zone in the north, an Italian occupation zone and an unoccupied territory, the rest of France, which consisted of southern France and the French empire. The Vichy government, an authoritarian regime collaborating with Germany, ruled the unoccupied territory. Free France, the government-in-exile led by Charles de Gaulle, was set up in London.[63] From 1942 to 1944, about 160,000 French ci...

[Chunk 1] Score: N/A
In 1940, France was invaded and quickly defeated by Nazi Germany. France was divided into a German occupation zone in the north, an Italian occupation zone and an unoccupied territory, the rest of France, which consisted of southern France and the French empire. The Vichy government, an authoritarian regime collaborating with Germany, ruled the unoccupied territory. Free France, the government-in-

[Chunk 2]
of the Third French Repub